# Importing Required Libraries

In [1]:
import os
import shutil
import random
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications.vgg16 import VGG16
from keras.layers import Dense,Flatten
from keras.models import Model
from keras.callbacks import EarlyStopping,ModelCheckpoint
import warnings
from keras.models import load_model
import numpy as np
from sklearn.metrics import classification_report,ConfusionMatrixDisplay,confusion_matrix
warnings.filterwarnings('ignore')

# Category

In [2]:
categories = set()
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        category = full_path.split('/')[-2][3:]
        categories.add(category)
categoryie= list(categories)
print(categories)

{'thumb', 'index', 'fist_moved', 'l', 'ok', 'down', 'c', 'palm_moved', 'palm', 'fist'}


# Preprocessing

In [3]:
augmentation=ImageDataGenerator(rescale=1.0/255,
                                rotation_range=10,
                                width_shift_range=0.1,
                                height_shift_range=0.1,
                                shear_range=0.1,
                                zoom_range=0.1,
                                horizontal_flip=True,
                                brightness_range=[0.9, 1.1],
                                channel_shift_range=0.01,
                                validation_split=0.2)

In [4]:
data_path="../input/leapgestrecog/leapGestRecog"
train_set=augmentation.flow_from_directory(data_path,
                                           target_size=(224,224),
                                           batch_size=32,
                                           class_mode='categorical',
                                           subset="training",
                                           shuffle=True)

validation_set=augmentation.flow_from_directory(data_path,
                                                target_size=(224,224),
                                                batch_size=32,
                                                class_mode='categorical',
                                                subset="validation",
                                                shuffle=False)

Found 16000 images belonging to 10 classes.
Found 4000 images belonging to 10 classes.


# VGG16 Model

In [5]:
image_size=[224,224]

In [6]:
vgg=VGG16(input_shape=image_size+[3],weights='imagenet',include_top=False)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
for layer in vgg.layers:
  layer.trainable=False

In [8]:
x=Flatten()(vgg.output)
output=Dense(10,activation='softmax')(x)

In [9]:
model=Model(inputs=vgg.input,outputs=output)

In [10]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │       250,890 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,965,578 (57.09 MB)

 Trainable params: 250,890 (980.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

# Fit and Train

In [ ]:
model_filepath = "best_model.keras"
early_stopping = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
model_checkpoint = ModelCheckpoint(filepath=model_filepath,
                                   monitor="val_loss",
                                   save_best_only=True,
                                   mode="min",
                                   verbose=1)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_set,
                              validation_data=validation_set,
                              epochs=2,
                              callbacks=[early_stopping,model_checkpoint])

Epoch 1/2
213/500 ━━━━━━━━━━━━━━━━━━━━ 44:46 9s/step - accuracy: 0.6063 - loss: 1.3133

In [ ]:
best_epoch_index=history.history['val_loss'].index(min(history.history['val_loss']))
best_epoch_index

In [ ]:
print("Train Loss:",history.history['loss'][0])
print("Train Accuracy:",history.history['accuracy'][0])
print("Validation Loss:",history.history['val_loss'][0])
print("Validation Accuracy:",history.history['val_accuracy'][0])

In [ ]:
labels=validation_set.classes
y_pred=model.predict(validation_set)

In [ ]:
y_pred = np.argmax(y_pred,axis=1)
print(ConfusionMatrixDisplay.from_predictions(labels,y_pred))
print(classification_report(labels,y_pred))

In [ ]:
plt.plot(history.history['accuracy'],label='Train Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curve')
plt.show()

In [ ]:
plt.plot(history.history['loss'],label='Train Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curve')
plt.show()